# Dataset Caching 📦

Pre-tokenize a dataset offline so training starts instantly.

**How it works:** Isla checks for a `*_tokenized/` folder next to your data file.
If found, it loads directly (skipping tokenization). If not, it tokenizes and saves automatically.

**v3 dataset:** Trilingual (45% EN, 30% PT, 15% Code, 10% Math) — ~1.5B tokens.

In [ ]:
!pip install -q datasets transformers

## 1. Config

In [ ]:
SOURCE_FILE = "./data/corpus_v3.jsonl"         # your raw dataset
OUTPUT_DIR = "./data/corpus_v3_tokenized"       # where to save
TOKENIZER_NAME = "NousResearch/Llama-2-7b-hf"  # v3 tokenizer
MAX_SEQ_LEN = 2048                              # v3 context length

## 2. Tokenize

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
from functools import partial

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer: {TOKENIZER_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
raw_ds = load_dataset("json", data_files={"train": SOURCE_FILE})
print(f"Loaded {len(raw_ds['train']):,} examples")
print(f"Columns: {raw_ds['train'].column_names}")

In [ ]:
def tokenize_batch(examples, tokenizer, max_length):
    texts = examples.get("text") or examples.get("content")
    if texts is None:
        raise ValueError(f"No text column. Available: {list(examples.keys())}")

    enc = tokenizer(texts, max_length=max_length, truncation=True,
                    padding="max_length", return_tensors=None)

    labels = [
        [tok_id if mask == 1 else -100 for tok_id, mask in zip(ids, attn)]
        for ids, attn in zip(enc["input_ids"], enc["attention_mask"])
    ]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

fn = partial(tokenize_batch, tokenizer=tokenizer, max_length=MAX_SEQ_LEN)
tokenized_ds = raw_ds.map(fn, batched=True, remove_columns=raw_ds["train"].column_names,
                          num_proc=4, desc="Tokenizing")

print(f"Tokenized {len(tokenized_ds['train']):,} examples")

## 3. Split & Save

In [ ]:
if "validation" not in tokenized_ds:
    split = tokenized_ds["train"].train_test_split(test_size=0.001, shuffle=True, seed=42)
    tokenized_ds = DatasetDict({"train": split["train"], "validation": split["test"]})
    print(f"Validation split: {len(tokenized_ds['validation']):,} samples")

tokenized_ds.save_to_disk(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}/")

## 4. Use in Training

Just point `DataConfig` to the tokenized folder or the original file (Isla finds the cache automatically):

```python
import isla

# option A: point to tokenized dir
data_config = isla.DataConfig(dataset_path="./data/corpus_v3_tokenized")

# option B: point to raw file (isla finds corpus_v3_tokenized/ automatically)
data_config = isla.DataConfig(dataset_path="./data/corpus_v3.jsonl")
```